# Hot zones Uber à New York

## Clustering géospatial non supervisé pour l'aide au positionnement des chauffeurs

**Objectif du projet**  
Construire une approche de clustering non supervisé pour identifier les **zones chaudes de pickups Uber à New York**, comparer **KMeans** et **DBSCAN**, puis transformer les résultats en **recommandations opérationnelles** lisibles.

**Fil directeur**  
Le notebook suit une progression simple et défendable :
1. comprendre le besoin métier ;  
2. charger et préparer les données ;  
3. analyser les rythmes de demande ;  
4. comparer deux approches de clustering ;  
5. cartographier les hot zones ;  
6. conclure sur l'usage le plus pertinent.


## 1. Contexte business

Le problème à traiter est simple : **les chauffeurs ne sont pas toujours au bon endroit au bon moment**.  
Quand la demande augmente dans une zone alors que l'offre est dispersée ailleurs, le temps d'attente s'allonge et la qualité de service diminue.

L'objectif est donc de répondre à une question concrète :

> **Où les chauffeurs devraient-ils se positionner selon le jour et l'heure ?**

Le projet vise à :
- identifier des **zones chaudes** de demande ;
- les **visualiser sur carte** ;
- comparer au moins **deux algorithmes non supervisés** ;
- restituer les résultats **au minimum par jour de semaine**.

La logique retenue est la suivante :
1. comprendre les données ;  
2. nettoyer et préparer la géographie ;  
3. repérer les moments les plus structurants ;  
4. comparer **KMeans** et **DBSCAN** ;  
5. produire des **cartes par jour** ;  
6. formuler une **recommandation opérationnelle**.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import zipfile

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors



## 2. Chargement des données

Le dataset fourni par le projet contient plusieurs fichiers mensuels de pickups Uber sur New York en 2014.  
Pour rester fidèle au brief, on se concentre sur les **fichiers 2014 d'avril à septembre**.

Le code ci-dessous charge directement les CSV depuis le zip, ce qui évite d'extraire manuellement les fichiers.


In [ ]:

# --- Paths ---
DATA_ROOT = Path(".")
ZIP_PATH = DATA_ROOT / "uber-trip-data.zip"

if not ZIP_PATH.exists():
    # fallback when the notebook is stored next to the outer project zip structure
    candidates = list(Path(".").rglob("uber-trip-data.zip"))
    if not candidates:
        raise FileNotFoundError("Impossible de trouver 'uber-trip-data.zip'. Place le notebook dans le dossier du projet ou à côté du zip.")
    ZIP_PATH = candidates[0]

print(f"Fichier détecté : {ZIP_PATH}")

# --- Load monthly raw data from the zip ---
monthly_frames = []
with zipfile.ZipFile(ZIP_PATH) as zf:
    raw_csvs = sorted(
        [name for name in zf.namelist()
         if name.endswith(".csv")
         and "uber-raw-data-" in name
         and "__MACOSX" not in name
         and "janjune15" not in name]
    )
    print("Fichiers mensuels utilisés :")
    for name in raw_csvs:
        print(" -", name)
        with zf.open(name) as f:
            df_month = pd.read_csv(
                f,
                parse_dates=["Date/Time"],
                usecols=["Date/Time", "Lat", "Lon", "Base"]
            )
            df_month["source_file"] = Path(name).name
            monthly_frames.append(df_month)

uber = pd.concat(monthly_frames, ignore_index=True)

print("\\nDimensions initiales :", uber.shape)
display(uber.head())



## 3. Nettoyage, bornes géographiques et variables temporelles

Comme le sujet parle de **New York**, on retire les points géographiques manifestement hors périmètre.  
C'est important pour deux raisons :

- éviter que quelques coordonnées aberrantes déforment les clusters ;
- rester cohérent avec l'objectif métier réel.

On ajoute ensuite des variables utiles pour l'analyse :
- **heure** ;
- **jour du mois** ;
- **jour de semaine** ;
- **week-end ou non**.


In [ ]:
# --- Bornes géographiques de New York (filtre volontairement conservateur) ---
NYC_BOUNDS = {
    "lat_min": 40.55,
    "lat_max": 40.92,
    "lon_min": -74.15,
    "lon_max": -73.70
}

mask_nyc = (
    uber["Lat"].between(NYC_BOUNDS["lat_min"], NYC_BOUNDS["lat_max"])
    & uber["Lon"].between(NYC_BOUNDS["lon_min"], NYC_BOUNDS["lon_max"])
)

uber_clean = uber.loc[mask_nyc].copy()

# --- Variables temporelles ---
weekday_map = {
    "Monday": "Lundi",
    "Tuesday": "Mardi",
    "Wednesday": "Mercredi",
    "Thursday": "Jeudi",
    "Friday": "Vendredi",
    "Saturday": "Samedi",
    "Sunday": "Dimanche"
}

time_block_map = {
    "Nuit": "Nuit",
    "Matin": "Matin",
    "Milieu de journée": "Milieu de journée",
    "Pic du soir": "Pic du soir",
    "Fin de soirée": "Fin de soirée"
}

uber_clean["hour"] = uber_clean["Date/Time"].dt.hour
uber_clean["day"] = uber_clean["Date/Time"].dt.day
uber_clean["weekday_num"] = uber_clean["Date/Time"].dt.weekday
uber_clean["weekday"] = uber_clean["Date/Time"].dt.day_name().map(weekday_map)
uber_clean["month"] = uber_clean["Date/Time"].dt.month_name()
uber_clean["is_weekend"] = uber_clean["weekday_num"] >= 5
uber_clean["time_block"] = pd.cut(
    uber_clean["hour"],
    bins=[-1, 5, 11, 16, 20, 23],
    labels=["Nuit", "Matin", "Milieu de journée", "Pic du soir", "Fin de soirée"]
)

weekday_order = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi", "Samedi", "Dimanche"]
time_block_order = ["Nuit", "Matin", "Milieu de journée", "Pic du soir", "Fin de soirée"]

print("Dimensions après filtre géographique :", uber_clean.shape)
print("Points retirés :", len(uber) - len(uber_clean))
display(uber_clean.head())



### Lecture rapide qualité / volume

Avant de clusteriser, on vérifie que l'on a un dataset suffisamment riche et bien réparti.  
Ici, le volume est très important, ce qui est excellent pour la robustesse analytique, mais cela impose aussi une stratégie **efficiente** pour les algorithmes.


In [ ]:

summary = pd.DataFrame({
    "metric": [
        "Total pickups (brut)",
        "Total pickups (NYC filtré)",
        "Nombre de bases Uber",
        "Latitude min",
        "Latitude max",
        "Longitude min",
        "Longitude max"
    ],
    "value": [
        len(uber),
        len(uber_clean),
        uber_clean["Base"].nunique(),
        round(uber_clean["Lat"].min(), 5),
        round(uber_clean["Lat"].max(), 5),
        round(uber_clean["Lon"].min(), 5),
        round(uber_clean["Lon"].max(), 5)
    ]
})
display(summary)



## 4. Analyse exploratoire

Le clustering seul ne suffit pas : pour que le projet soit convaincant, il faut d'abord comprendre **quand** la demande se concentre.  
Cette partie permet d'identifier les moments stratégiques où des hot zones sont particulièrement utiles.

Nous allons regarder :
- la charge par **jour de semaine** ;
- la charge par **heure** ;
- la différence **semaine vs week-end** ;
- les créneaux les plus pertinents pour lancer une première expérimentation.


In [ ]:
weekday_counts = (
    uber_clean["weekday"]
    .value_counts()
    .reindex(weekday_order)
    .reset_index()
)
weekday_counts.columns = ["weekday", "pickups"]

fig = px.bar(
    weekday_counts,
    x="weekday",
    y="pickups",
    title="Nombre de pickups par jour de semaine",
    text_auto=True
)
fig.update_layout(xaxis_title="Jour", yaxis_title="Nombre de pickups")
fig.show()

hour_counts = (
    uber_clean.groupby("hour")
    .size()
    .reset_index(name="pickups")
)

fig = px.line(
    hour_counts,
    x="hour",
    y="pickups",
    markers=True,
    title="Intensité horaire de la demande Uber"
)
fig.update_layout(xaxis_title="Heure", yaxis_title="Nombre de pickups")
fig.show()

weekend_counts = (
    uber_clean.groupby("is_weekend")
    .size()
    .rename(index={False: "Semaine", True: "Week-end"})
    .reset_index(name="pickups")
)
fig = px.bar(
    weekend_counts,
    x="is_weekend",
    y="pickups",
    title="Comparaison semaine / week-end",
    text_auto=True
)
fig.update_layout(xaxis_title="Type de jour", yaxis_title="Nombre de pickups")
fig.show()

time_block_counts = (
    uber_clean.groupby("time_block", observed=False)
    .size()
    .reindex(time_block_order)
    .reset_index(name="pickups")
)
fig = px.bar(
    time_block_counts,
    x="time_block",
    y="pickups",
    title="Répartition par grand moment de la journée",
    text_auto=True
)
fig.update_layout(xaxis_title="Moment de la journée", yaxis_title="Nombre de pickups")
fig.show()


## 5. Stratégie méthodologique : commencer petit, puis généraliser

Une démarche raisonnable consiste à **commencer sur un créneau simple et dense**, puis à étendre l'approche.

Nous allons donc :

1. choisir un **créneau de référence** suffisamment riche pour être informatif ;  
2. y comparer **KMeans** et **DBSCAN** ;  
3. retenir des paramètres cohérents ;  
4. généraliser ensuite l'approche à chaque jour de semaine.

### Choix du créneau de référence
Pour un premier test, un créneau de **fort trafic en semaine** est un bon point de départ, par exemple :
- **jeudi** ;
- entre **17 h et 20 h**.

Ce type de fenêtre concentre :
- des volumes élevés ;
- des flux domicile-travail ;
- une structure spatiale suffisamment marquée pour faire apparaître des hot zones lisibles.


In [ ]:
reference_days = ["Jeudi"]
reference_hours = [17, 18, 19, 20]

ref_df = uber_clean[
    uber_clean["weekday"].isin(reference_days)
    & uber_clean["hour"].isin(reference_hours)
].copy()

print("Taille du créneau de référence :", ref_df.shape)
display(ref_df.head())

fig = px.scatter_mapbox(
    ref_df.sample(min(15000, len(ref_df)), random_state=42),
    lat="Lat",
    lon="Lon",
    zoom=9.7,
    height=650,
    title="Créneau de référence : jeudi 17 h - 20 h (échantillon visuel)"
)
fig.update_layout(mapbox_style="carto-positron", margin=dict(l=0, r=0, t=50, b=0))
fig.show()



## 6. Fonctions utilitaires

Pour garder un notebook propre, on crée quelques fonctions réutilisables :
- préparation des coordonnées ;
- tuning de **KMeans** ;
- tuning de **DBSCAN** ;
- création de cartes lisibles.


In [ ]:

def prepare_geo_matrix(df, columns=("Lat", "Lon")):
    X = df.loc[:, columns].copy()
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X, X_scaled, scaler

def tune_kmeans(X_scaled, k_values=range(3, 11), random_state=42):
    results = []
    for k in k_values:
        model = MiniBatchKMeans(
            n_clusters=k,
            random_state=random_state,
            n_init=10,
            batch_size=2048
        )
        labels = model.fit_predict(X_scaled)
        sil = silhouette_score(
            X_scaled,
            labels,
            sample_size=min(3000, len(X_scaled)),
            random_state=random_state
        )
        results.append({
            "k": k,
            "inertia": model.inertia_,
            "silhouette": sil
        })
    return pd.DataFrame(results)

def tune_dbscan(X_scaled, eps_values, min_samples=80):
    rows = []
    for eps in eps_values:
        model = DBSCAN(eps=eps, min_samples=min_samples, n_jobs=1)
        labels = model.fit_predict(X_scaled)

        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        noise_ratio = np.mean(labels == -1)

        silhouette = np.nan
        non_noise_mask = labels != -1
        if n_clusters >= 2 and non_noise_mask.sum() > 500:
            silhouette = silhouette_score(
                X_scaled[non_noise_mask],
                labels[non_noise_mask],
                sample_size=min(2000, non_noise_mask.sum()),
                random_state=42
            )

        rows.append({
            "eps": eps,
            "min_samples": min_samples,
            "n_clusters": n_clusters,
            "noise_ratio": noise_ratio,
            "silhouette_non_noise": silhouette
        })
    return pd.DataFrame(rows)

def plot_kmeans_centroids(df, labels, centers_latlon, title):
    plot_df = df.copy()
    plot_df["cluster"] = labels.astype(str)

    centers_df = pd.DataFrame(centers_latlon, columns=["Lat", "Lon"])
    centers_df["cluster"] = centers_df.index.astype(str)

    fig = go.Figure()

    # Light sample of raw points for context
    sampled_points = plot_df.sample(min(8000, len(plot_df)), random_state=42)
    fig.add_trace(go.Scattermapbox(
        lat=sampled_points["Lat"],
        lon=sampled_points["Lon"],
        mode="markers",
        marker={"size": 5, "opacity": 0.25},
        text=sampled_points["cluster"],
        name="Pickups sample"
    ))

    fig.add_trace(go.Scattermapbox(
        lat=centers_df["Lat"],
        lon=centers_df["Lon"],
        mode="markers+text",
        marker={"size": 16},
        text=[f"C{i}" for i in centers_df.index],
        textposition="top right",
        name="KMeans centroids"
    ))

    fig.update_layout(
        mapbox_style="carto-positron",
        mapbox_zoom=9.7,
        mapbox_center={"lat": 40.74, "lon": -73.97},
        height=650,
        title=title,
        margin=dict(l=0, r=0, t=50, b=0)
    )
    return fig

def plot_dbscan_clusters(df, labels, title):
    plot_df = df.copy()
    plot_df["cluster"] = labels.astype(str)

    # keep a readable sample for display
    sampled = plot_df.sample(min(12000, len(plot_df)), random_state=42)

    fig = px.scatter_mapbox(
        sampled,
        lat="Lat",
        lon="Lon",
        color="cluster",
        zoom=9.7,
        height=650,
        title=title,
        opacity=0.55
    )
    fig.update_layout(mapbox_style="carto-positron", margin=dict(l=0, r=0, t=50, b=0))
    return fig

def cluster_summary(df, label_col="cluster"):
    out = (
        df.groupby(label_col)
        .agg(
            pickups=("Lat", "size"),
            mean_lat=("Lat", "mean"),
            mean_lon=("Lon", "mean")
        )
        .sort_values("pickups", ascending=False)
        .reset_index()
    )
    return out


## 7. KMeans — tuning et interprétation

### Pourquoi KMeans ?
KMeans correspond bien à une logique **opérationnelle** :
- il force une partition claire de l'espace ;
- il produit des **centroïdes** faciles à interpréter ;
- il reste lisible pour une équipe métier.

Sa limite est connue : il suppose des clusters plutôt compacts et impose un nombre de clusters `k`.  
Mais pour une logique de **zones de positionnement chauffeurs**, cette contrainte peut aussi devenir un avantage : il faut justement aboutir à des zones simples et pilotables.


In [ ]:

# Optional sampling to keep execution efficient on laptops / notebooks
# Échantillon de travail pour garder une exécution fluide sur une machine standard
ref_sample = ref_df.sample(min(10000, len(ref_df)), random_state=42).copy()
X_ref, X_ref_scaled, scaler_ref = prepare_geo_matrix(ref_sample)

kmeans_tuning = tune_kmeans(X_ref_scaled, k_values=range(3, 11))
display(kmeans_tuning)

fig = go.Figure()
fig.add_trace(go.Scatter(x=kmeans_tuning["k"], y=kmeans_tuning["inertia"], mode="lines+markers", name="Inertia"))
fig.update_layout(title="KMeans — courbe elbow", xaxis_title="k", yaxis_title="Inertia")
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=kmeans_tuning["k"], y=kmeans_tuning["silhouette"], mode="lines+markers", name="Silhouette"))
fig.update_layout(title="KMeans — score de silhouette", xaxis_title="k", yaxis_title="Silhouette")
fig.show()

# Heuristic choice:
best_k = int(kmeans_tuning.sort_values(["silhouette", "k"], ascending=[False, True]).iloc[0]["k"])
print("k retenu automatiquement (silhouette max) :", best_k)

kmeans_model = MiniBatchKMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=20,
    batch_size=2048
)
kmeans_labels = kmeans_model.fit_predict(X_ref_scaled)

# Bring centroids back to original latitude/longitude space
centers_latlon = scaler_ref.inverse_transform(kmeans_model.cluster_centers_)

ref_kmeans = ref_sample.copy()
ref_kmeans["cluster"] = kmeans_labels.astype(int)

display(cluster_summary(ref_kmeans, "cluster"))

fig = plot_kmeans_centroids(
    ref_sample,
    kmeans_labels,
    centers_latlon,
    title=f"KMeans — hot zones sur le créneau de référence (jeudi 17 h - 20 h), k={best_k}"
)
fig.show()



## 8. DBSCAN — tuning et interprétation

### Pourquoi DBSCAN ?
DBSCAN apporte une logique différente :
- il détecte les **zones denses** ;
- il peut gérer des formes moins “sphériques” ;
- il identifie explicitement le **bruit** (`-1`), ce qui est intéressant pour distinguer les zones très actives des pickups plus isolés.

En revanche, DBSCAN est plus sensible à ses paramètres :
- `eps` : rayon de voisinage ;
- `min_samples` : densité minimale pour former un cluster.

C'est un bon benchmark analytique, mais il n'est pas toujours le plus simple à traduire en zones métier stables.


In [ ]:

# k-distance helper for eps intuition
neighbors = NearestNeighbors(n_neighbors=80)
neighbors_fit = neighbors.fit(X_ref_scaled)
distances, indices = neighbors_fit.kneighbors(X_ref_scaled)
k_distance = np.sort(distances[:, -1])

fig = px.line(
    x=np.arange(len(k_distance)),
    y=k_distance,
    title="DBSCAN — courbe k-distance (80e voisin)"
)
fig.update_layout(xaxis_title="Observations triées", yaxis_title="Distance au 80e voisin")
fig.show()

eps_grid = [0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.10]
dbscan_tuning = tune_dbscan(X_ref_scaled, eps_values=eps_grid, min_samples=80)
display(dbscan_tuning)

# pragmatic choice:
dbscan_choice = (
    dbscan_tuning
    .query("n_clusters >= 2")
    .sort_values(["silhouette_non_noise", "noise_ratio"], ascending=[False, True])
    .head(1)
)

if len(dbscan_choice) == 0:
    chosen_eps = 0.06
else:
    chosen_eps = float(dbscan_choice.iloc[0]["eps"])

print("eps retenu automatiquement :", chosen_eps)

dbscan_model = DBSCAN(eps=chosen_eps, min_samples=80, n_jobs=1)
dbscan_labels = dbscan_model.fit_predict(X_ref_scaled)

ref_dbscan = ref_sample.copy()
ref_dbscan["cluster"] = dbscan_labels

display(cluster_summary(ref_dbscan.query("cluster != -1"), "cluster").head(15))
print("Part du bruit :", round((ref_dbscan['cluster'] == -1).mean(), 4))

fig = plot_dbscan_clusters(
    ref_sample,
    dbscan_labels,
    title=f"DBSCAN — hot zones sur le créneau de référence (jeudi 17 h - 20 h), eps={chosen_eps}, min_samples=80"
)
fig.show()


## 9. Comparaison KMeans vs DBSCAN

Cette partie vise à comparer les deux algorithmes **dans le cadre précis du besoin Uber**.

L'enjeu n'est pas de désigner un algorithme “meilleur” dans l'absolu, mais d'identifier celui qui fournit la restitution la plus **utile pour la décision**.

### Lecture attendue
- **KMeans** : pertinent pour produire un **petit nombre de zones stables** et faciles à piloter ;
- **DBSCAN** : pertinent pour repérer les **poches de densité** et isoler le bruit, mais parfois moins simple à transformer en consignes opérationnelles.


In [ ]:

comparison = pd.DataFrame({
    "Dimension": [
        "Logique",
        "Paramètres clés",
        "Forme des clusters",
        "Gestion du bruit",
        "Lisibilité métier",
        "Scalabilité",
        "Utilité pour Uber"
    ],
    "KMeans": [
        "Partitionne l'espace en k zones",
        "Nombre de clusters k",
        "Plutôt compacts / convexes",
        "Pas de bruit explicite",
        "Très lisible via centroïdes",
        "Très bon avec MiniBatchKMeans",
        "Très bon pour recommander des zones stables"
    ],
    "DBSCAN": [
        "Détecte des zones denses",
        "eps + min_samples",
        "Peut suivre des formes irrégulières",
        "Oui, bruit = label -1",
        "Moins simple à communiquer",
        "Plus sensible au volume et aux paramètres",
        "Très bon pour analyser la densité réelle"
    ]
})
display(comparison)


## 10. Généralisation à chaque jour de semaine

Le projet demande **au minimum des hot zones par jour de semaine**.  
On étend donc maintenant l'approche au niveau hebdomadaire.

### Choix pratique retenu
Pour garder un notebook :
- robuste ;
- exécutable ;
- lisible ;

on conserve une fenêtre homogène et pertinente :
- **17 h à 20 h** ;
- pour **chaque jour de la semaine** ;
- avec **KMeans** comme méthode principale de restitution, car c'est la plus claire côté métier.


In [ ]:

def fit_daily_kmeans(df, weekday, hours, k):
    day_df = df[(df["weekday"] == weekday) & (df["hour"].isin(hours))].copy()
    if len(day_df) == 0:
        return None, None, None

    day_sample = day_df.sample(min(12000, len(day_df)), random_state=42)
    X_day, X_day_scaled, scaler_day = prepare_geo_matrix(day_sample)

    model = MiniBatchKMeans(
        n_clusters=k,
        random_state=42,
        n_init=15,
        batch_size=2048
    )
    labels = model.fit_predict(X_day_scaled)
    centers_latlon = scaler_day.inverse_transform(model.cluster_centers_)

    result = day_sample.copy()
    result["cluster"] = labels.astype(int)
    return result, centers_latlon, model

daily_results = {}
for day in weekday_order:
    result, centers, model = fit_daily_kmeans(uber_clean, day, [17, 18, 19, 20], best_k)
    daily_results[day] = {
        "data": result,
        "centers": centers,
        "model": model
    }

for day in weekday_order:
    payload = daily_results[day]
    if payload["data"] is None:
        print(f"{day}: aucune donnée")
        continue

    fig = plot_kmeans_centroids(
        payload["data"],
        payload["data"]["cluster"].values,
        payload["centers"],
        title=f"{day} — hot zones KMeans (17 h - 20 h)"
    )
    fig.show()


## 11. Carte synthétique des centres par jour

Une fois les centres calculés pour chaque jour, il est utile de les rassembler dans une seule vue.  
Cette carte permet de comparer visuellement les zones récurrentes de demande et de repérer les secteurs les plus stables.


In [ ]:

all_centers = []

for day in weekday_order:
    payload = daily_results[day]
    if payload["centers"] is None:
        continue
    centers_df = pd.DataFrame(payload["centers"], columns=["Lat", "Lon"])
    centers_df["weekday"] = day
    centers_df["cluster"] = [f"C{i}" for i in range(len(centers_df))]
    all_centers.append(centers_df)

all_centers_df = pd.concat(all_centers, ignore_index=True)

fig = px.scatter_mapbox(
    all_centers_df,
    lat="Lat",
    lon="Lon",
    color="weekday",
    hover_data=["cluster"],
    zoom=9.7,
    height=700,
    title="Synthèse — centres des hot zones par jour de semaine (17 h - 20 h)"
)
fig.update_traces(marker={"size": 14})
fig.update_layout(mapbox_style="carto-positron", margin=dict(l=0, r=0, t=50, b=0))
fig.show()


## 12. Lecture business des résultats

Même si la position exacte des centres peut varier légèrement selon l'échantillonnage et les paramètres, plusieurs constats ressortent :

- le cœur de la demande se concentre très fortement sur **Manhattan** ;
- des zones secondaires récurrentes apparaissent vers **Brooklyn**, **Queens** et certains **axes de transit** ;
- la structure spatiale est plus marquée sur les jours de semaine en **fin de journée**, ce qui est cohérent avec les flux de mobilité urbaine.

### Ce que cela signifie
Ces résultats peuvent aider à :
- pré-positionner les chauffeurs avant les pics ;
- adapter des recommandations par **jour** et par **créneau horaire** ;
- préparer des enrichissements futurs avec d'autres signaux : météo, événements, trafic, aéroports, etc.


## 13. Conclusion

### Ce que montre le projet
Un pipeline simple de **clustering non supervisé** permet déjà de transformer des historiques de pickups en **zones d'action concrètes**.

### Recommandation principale
- **KMeans** est la méthode la plus adaptée ici comme base de restitution, car elle fournit des **zones stables, lisibles et directement exploitables**.
- **DBSCAN** constitue un complément utile pour **repérer les noyaux de densité** et distinguer le **bruit**, mais il est moins simple à convertir en consignes régulières de positionnement.

### Réponse au besoin
Le cahier des charges est couvert :
- carte de hot zones ;
- comparaison d'au moins deux algorithmes non supervisés ;
- lecture par jour de semaine ;
- restitution visuelle exploitable.


## 14. Limites et pistes d'amélioration

### Limites
- le dataset couvre une période donnée en 2014 ;
- l'analyse repose ici uniquement sur la **géolocalisation des pickups** ;
- les résultats dépendent en partie des paramètres de clustering ;
- le modèle ne prend pas encore en compte des variables externes telles que :
  - météo,
  - événements,
  - trafic,
  - typologie de quartier.

### Pistes d'amélioration
- passer d'un bloc horaire à une analyse **heure par heure** ;
- enrichir l'approche avec des données contextuelles ;
- projeter les clusters sur des **zones opérationnelles fixes** ;
- explorer une logique de recommandation plus proche du temps réel.


## 15. Synthèse courte

> **KMeans fournit ici la meilleure base de restitution opérationnelle, car il transforme une géographie complexe en zones simples et lisibles. DBSCAN complète l'analyse en mettant en évidence les noyaux de densité et le bruit.**